# Leave-one-out solver analysis — window write-up

What would the CoW Protocol solver competition have looked like without a given
solver? This report replays a window of recorded auctions with one solver's bids
removed: winner selection is re-run on the remaining bids and the outcome is compared
with what actually happened. Three effects are measured per solver:

- **user surplus** — how much better or worse off users would have been,
- **solver rewards** — how much the protocol would have paid the remaining solvers,
- **order coverage** — which orders would not have traded at all.

**Reading the signs:** every delta is **counterfactual − actual**, the world *without*
the solver minus the world *with* it. Negative Δsurplus means users would have
received less without the solver; positive Δrewards means the protocol would have
paid more.

Everything below is post-processing of the per-auction reports written by
`loo analyse`; the only database access is the stablecoin prices behind the USD
columns, and the notebook also runs without it (USD columns are then omitted). The
inputs are generated once per network — solvers share one extraction pass, and each
run is repeated for the two settlement scenarios explained below:

```bash
for rule in inherited assume-settled; do
  uv run loo analyse --network mainnet \
      --solver Baseline --solver Rizzolver \
      --start 2026-07-01 --end 2026-08-01 \
      --outcome-rule $rule --out "out/jul2026-mainnet-{solver}-$rule.json"
done
```

The same table on the command line: `uv run loo compare out/jul2026-*.json`.
How the pipeline works and how it is validated against the recorded competition:
[README](../README.md). Background and design history: [PLAN.md](../PLAN.md),
[docs/winner-selection.md](../docs/winner-selection.md),
[docs/rewards.md](../docs/rewards.md),
[docs/analytics-db.md](../docs/analytics-db.md).


In [ ]:
from pathlib import Path

import pandas as pd

from loo import aggregate

# Which analyse reports to aggregate — adjust the glob to pick the window to report on.
REPORT_DIR = Path("..") / "out"
REPORT_GLOB = "jul2026-*.json"

reports = [aggregate.load_report(str(p)) for p in sorted(REPORT_DIR.glob(REPORT_GLOB))]
if not reports:
    raise FileNotFoundError(
        f"no analyse reports matching {REPORT_GLOB} in {REPORT_DIR.resolve()} — "
        f"generate them with the commands in the cell above"
    )
windows = aggregate.group_reports(reports)

pd.DataFrame(
    {
        "solver": r.solver,
        "window": f"{r.start}..{r.end}",
        "network": r.network,
        "mode": r.mode,
        "outcome rule": r.outcome_rule,
        "auctions analysed": r.analysed,
        "changed auctions": len(r.moves),
        "file": Path(r.path).name,
    }
    for r in reports
)


In [ ]:
# USD rates, implied per auction by the median stablecoin price in the auction's own
# price vector (D15). Skipped gracefully when the DB is unreachable.
usd_by_network = {}
try:
    from loo import db, extract

    for network in sorted({r.network for r in reports}):
        auction_ids = sorted(
            {m.auction_id for r in reports if r.network == network for m in r.moves}
        )
        conn = db.connect(network)
        try:
            context = aggregate.usd_context(
                extract.load_usd_rates(conn, auction_ids, network)
            )
        finally:
            conn.close()
        if context is not None:
            usd_by_network[network] = context
            print(
                f"{network}: rates for {len(context.rates)} auctions, "
                f"window median {aggregate.usd_amount(context.fallback)}/native"
            )
except Exception as error:  # offline is a supported mode, not a failure
    print(f"USD conversion skipped ({type(error).__name__}: {error})")

## The comparison

One column per solver and network. The first rows say how often the solver bid and
won; the Δ rows say what its removal would have changed. Two things to know before
reading them:

**The two settlement scenarios.** Winning an auction does not guarantee the batch
lands on-chain — on the mainnet calibration window, 14.9% of recorded winners never
settled in time, and they carried half of the winning score. A counterfactual winner
never ran at all, so the analysis has to assume something about settlement, and
rather than hiding one assumption in the code it reports two defensible readings:

- **`inherited`** — the headline. Settlement belongs to the auction slot, not the
  solver: a replacement winner inherits the recorded outcome of the winner it
  replaces, so a batch that actually reverted stays reverted whoever wins it. This is
  the scenario grounded in the record.
- **`assume-settled`** — every winner, recorded or replacement, is assumed to land in
  time, so the comparison is about proposals alone and settlement risk is excluded.
  The gap between the two readings is genuine uncertainty about whether failed
  batches would have landed.

**The two reward figures.** *Uncapped* is the reward mechanism's exact accounting,
but not what is paid out — failed settlements incur uncapped penalties far below the
real payout floor. *Capped (estimate)* clamps every reward into the recorded payout
caps and is the payout-scale answer, but an estimate: a replacement winner inherits
the reward cap of the slot it takes. The net-change row nets Δsurplus against the
capped figure.


In [ ]:
table = aggregate.comparison(windows, usd_by_network)
print(f"signs: {aggregate.SIGN_CONVENTION}")
for warning in table.warnings:
    print(f"WARNING: {warning}")

frame = pd.DataFrame(
    {
        column: [cells[i] for _, cells in table.rows]
        for i, column in enumerate(table.columns)
    },
    index=[label for label, _ in table.rows],
)
frame.style.set_properties(**{"text-align": "left"})

### Caveats — these travel with every number above

1. **No behavioural response.** The remaining solvers' bids are held fixed; nothing
   is claimed about how they would bid if the removed solver actually left. The
   competition's cap on solutions per solver is also applied before winners are
   picked, so a rival solution suppressed by that cap cannot step in either.
2. **Settlement risk is an assumption, and the headline names it.** `inherited`
   reads settlement off the record, `assume-settled` assumes everything lands in
   time (see above). With 14.9% of calibration-window winners never settling in time
   while carrying 50.2% of winning score, the choice is first-order, not a footnote.
3. **The fairness filter is approximated.** The competition discards solutions that
   are unfair on some token pair by comparing per-pair scores, which are not
   recorded anywhere; the filter is re-run on per-pair user surplus instead.
   Measured against the recorded filter decisions, this proxy changes 7 of 7,745
   auctions ([details](../docs/winner-selection.md#the-filter-runs-on-surplus)).
4. **Quote rewards are excluded** — there is no data on counterfactual quoting.
5. **Two reward figures, never one.** Uncapped is exact accounting but three orders
   of magnitude away from payouts (−410 ETH uncapped against 0.75 ETH actually paid
   on the calibration window); capped is the money answer but an estimate (see
   above).
6. **Auctions with suspect prices are excluded** and counted in the table. Native
   token prices in the auction data are occasionally wrong by orders of magnitude;
   every solution is cross-checked by valuing each trade through both tokens'
   prices, and an auction where the two sides disagree by more than 2× is dropped
   from every statistic. Before this check existed, one fabricated price supplied
   82% of a solver's headline.
7. **A small share of auctions cannot be replayed** and is excluded and counted in
   the table: solver-provided (JIT) orders are only recorded for batches that
   settled, so an unsettled solution's JIT orders are unrecoverable and its auction
   cannot be re-arbitrated faithfully. These exclusions are not random — they are
   JIT-heavy and reverted-winner auctions.
8. **USD figures are display conversions** at each auction's own stablecoin-implied
   rate; they inherit every caveat of the native-token figure they restate.


## Whales vs the median

Window totals are dominated by a handful of large auctions, so a sum alone is a
fragile summary — this is why medians and the largest single auction sit beside every
sum in the table, and why the largest movers are worth inspecting one by one.

The left plot shows how concentrated each headline is: the share of a window's total
|Δsurplus| carried by its N largest auctions. The right plot shows the distribution
of per-auction |Δsurplus| over the auctions that moved at all, pooled per solver.
Both are in USD so that windows on different chains are comparable, and both are
deliberately magnitude views (absolute values) — concentration and spread are
questions about size, and a log axis cannot carry a sign. The direction is in the
comparison table's `auctions moved + / −` row; on this window it is near-uniform:
almost every moved auction loses surplus in the counterfactual.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def usd_deltas(window):
    """|Δsurplus| per moved auction in USD, largest first; None without USD rates."""
    usd = usd_by_network.get(window.network)
    if usd is None:
        return None
    return np.array(
        sorted(
            (
                abs(m.delta_surplus) / 1e18 * float(usd.rate(m.auction_id))
                for m in window.headline.moves
                if m.delta_surplus
            ),
            reverse=True,
        )
    )


solvers = sorted({w.solver for w in windows})
colors = dict(
    zip(solvers, plt.rcParams["axes.prop_cycle"].by_key()["color"], strict=False)
)

fig, (concentration, histogram) = plt.subplots(1, 2, figsize=(12, 4))

labelled: set[str] = set()
skipped: list[str] = []
pooled: dict[str, list[np.ndarray]] = {solver: [] for solver in solvers}
for window in windows:
    deltas = usd_deltas(window)
    if deltas is None:
        skipped.append(f"{window.solver} ({window.network})")
        continue
    if not deltas.size:
        continue
    pooled[window.solver].append(deltas)
    concentration.plot(
        np.arange(1, deltas.size + 1),
        deltas.cumsum() / deltas.sum(),
        color=colors[window.solver],
        alpha=0.5,
        label=window.solver if window.solver not in labelled else "_nolegend_",
    )
    labelled.add(window.solver)

for solver, chunks in pooled.items():
    if chunks:
        histogram.hist(
            np.log10(np.concatenate(chunks)),
            bins=40,
            alpha=0.6,
            color=colors[solver],
            label=solver,
        )

concentration.set_xscale("log")
concentration.set_xlabel("N largest auctions")
concentration.set_ylabel("share of Σ|Δsurplus|")
concentration.set_title("concentration of |Δsurplus| (inherited) — one line per network")
concentration.legend()
histogram.set_xlabel("log10(|Δsurplus| in USD)")
histogram.set_ylabel("auctions")
histogram.set_title("per-auction |Δsurplus|, non-zero only")
histogram.legend()
fig.tight_layout()
if skipped:
    print("no USD rate, not plotted:", ", ".join(skipped))


In [ ]:
# The individually-inspectable tail: the ten auctions that move each solver's
# headline most, ranked in USD so different chains are comparable. The native
# column is in each row's own network token (ETH, xDAI, BNB, ...).
rows = []
for window in windows:
    usd = usd_by_network.get(window.network)
    for m in window.headline.moves:
        if not m.delta_surplus:
            continue
        rows.append(
            {
                "solver": window.solver,
                "network": window.network,
                "auction": m.auction_id,
                "Δsurplus [native]": m.delta_surplus / 1e18,
                "Δsurplus [USD]": (
                    m.delta_surplus / 1e18 * float(usd.rate(m.auction_id))
                    if usd is not None
                    else None
                ),
                "Δrewards uncapped [native]": m.delta_rewards / 1e18,
                "Δrewards capped [native]": (
                    float(m.delta_rewards_capped) / 1e18
                    if m.delta_rewards_capped is not None
                    else None
                ),
            }
        )
frame = pd.DataFrame(rows)
rank = (
    frame["Δsurplus [USD]"]
    if frame["Δsurplus [USD]"].notna().all()
    else frame["Δsurplus [native]"]
).abs()
(
    frame.assign(rank=rank)
    .sort_values(["solver", "rank"], ascending=[True, False])
    .groupby("solver")
    .head(10)
    .drop(columns="rank")
    .reset_index(drop=True)
    .round(6)
)


## How to read the result

- **Δsurplus** is the change in what users receive when the solver is removed, under
  the competition's own decisions and the named settlement scenario.
- **Δrewards capped (estimate)** is the change in payout-scale payments — quote this
  one as money. **Δrewards uncapped** is the change in the mechanism's internal
  accounting — quote it only as such.
- **Net change** = Δsurplus − capped Δrewards: what the removal scenario does to
  users and the protocol treasury combined.
- **Orders executed only with the solver** counts orders that traded in reality but
  that no other solver's solution executes in the counterfactual — the coverage
  answer. Its converse ("only without") is legitimately non-zero: removing a winner
  can unblock a rival's batch that executes an order the winner did not
  ([mechanism](../docs/winner-selection.md#a-blocked-batch-keeps-orders-unexecuted)).

The deltas describe the removal scenario under the caveats above; they are not by
themselves a valuation of a solver — surplus, rewards and coverage can legitimately
rank solvers differently, which is why they are reported side by side.
